# Buying fake followers: luck, strategy and fairness for content creators

On social media, creators compete for followers, and the platform's recommender system decides who gets
seen. Some creators can afford to buy fake followers to look more popular than they are. In this
assignment you simulate a platform to find out when buying pays off, who loses when someone buys, and
how the answer depends on the recommender.

## The platform

There are 20 content creators of different quality: creator 1 is the best and creator 20 the worst.
There are also 500 *seekers*, users looking for good creators to follow. At the start, nobody follows anyone.

In every round, the platform recommends one creator to each seeker. A seeker follows the recommended
creator if it is better than every creator they already follow, and never unfollows anyone. Over time,
seekers move up towards the best creators; how fast, and through which creators, depends on the recommender.
The recommender shows creator *i* with probability proportional to (1 + followers of *i*)<sup>α</sup>:

| recommender | α | what it does |
|---|---|---|
| AntiPA | −1 | favours unpopular creators |
| UR | 0 | uniform at random |
| PA | 1 | favours popular creators (preferential attachment) |
| ExtremePA | ∞ | only ever shows the most followed creator(s) |

The model comes from S. Ionescu, N. Pagan and A. Hannák, *Individual Fairness for Social Media Influencers*
(Complex Networks 2022) and *The role of luck in the success of social media influencers*
(Applied Network Science, 2023).

## What you will do

| part | question |
|---|---|
| 1 | Without fake followers, does quality decide success? |
| 2 | When does buying fake followers pay off for a creator? |
| 3 | Who pays for it: other creators, or seekers? |
| 4 | What happens when two creators can buy? |
| 5 | Which recommender should the platform choose? |

## What you hand in

**This notebook is not handed in or graded.** It is the simulation environment for the second half of
the Game Theory & Control Assignment 2 exercise (*Strategic Classification of Influencers*), which you
answer in your PDF on Moodle. Use it as a guide: its tasks and questions walk you through the
simulations those assignment questions rely on. We will not see your notebook, so put any figure,
table or number you use in your PDF.

| assignment question about | notebook part |
|---|---|
| how other creators and seekers are affected by one creator's strategic behaviour | 3 |
| the Nash equilibria when creators 4 and 11 can buy (budgets 25 / 25 and 5 / 50) | 4 |
| which α discourages buying fake followers | 5 (and 1–3) |
| changing an unrealistic model assumption | 5, Question 5.2, and the code in `src/fake_followers/` |

## Working in this notebook

Run the cells in order. Fill in each TODO before running the cells that depend on it, and write your
notes below the questions. Each part runs its simulations the first time you execute it, which can take
up to half a minute; the results are saved in `assignment/experiments/` and reused afterwards.

In [ ]:
import matplotlib.pyplot as plt

from fake_followers import experiments as ex
from fake_followers import plots

# the recommenders we compare: (rs_model, alpha)
RECOMMENDERS = [('general', -1), ('general', 0), ('general', 1), ('ExtremePA', 1000)]

NUM_INFLUENCERS = 5   # how many creators get paid
TOP_PAYOFF = 100      # what the most followed creator earns
NUM_ROUNDS = 100      # when payoffs are measured (one of 10, 50, 100)

## Part 1: The baseline (nobody can buy)

First, a platform where nobody can buy followers. Each recommender is simulated with 30 random seeds;
the runs differ only in luck.

In [ ]:
creators1, seekers1 = ex.run_experiment('part1.pkl', RECOMMENDERS, budget_profiles=[{}])
outcomes1 = ex.add_outcomes(creators1, NUM_INFLUENCERS, TOP_PAYOFF)
outcomes1.head()

`outcomes1` has one row per (run, recorded timestep, creator). The columns you need:

| column | meaning |
|---|---|
| `rs_model`, `alpha` | the recommender |
| `random_seed` | the run's seed (runs differ only in luck) |
| `timestep` | the round the row describes (10, 50 or 100) |
| `creator` | the creator's id: 0 is the best creator, so creator *k* has id *k* − 1 |
| `real_followers`, `total_followers` | real followers, and real + fake (the same here, since nobody buys) |
| `rank` | position by total followers (1 = most followed; tied creators share the better position) |
| `fair` | whether the outcome is fair for this creator (see below) |

**Fairness.** Creator *k* (the *k*-th best) has *quality rank* *k*, and a *popularity rank*: 1 + the number
of creators that end up with strictly more followers. The outcome is *fair* for creator *k* if its
popularity rank is at most its quality rank, that is, at most *k* − 1 creators end up ahead of it. Ties
count as fair. Worse creators may overtake creator *k*, as long as no more than *k* − 1 creators in total
end up ahead of it.

**Task 1.1.** Write `is_fair(followers)`. It gets the follower counts of one run, ordered by quality
(`followers[0]` belongs to the best creator), and returns one `True`/`False` per creator: whether the
outcome is fair for that creator.

In [ ]:
def is_fair(followers):
    '''followers[i]: followers of the creator with id i (0 = best). Returns one True/False per creator.'''
    # TODO
    ...

In [ ]:
assert is_fair([1]) is not None, 'is_fair returns nothing yet: complete Task 1.1 first'
assert list(is_fair([10, 5, 7, 10])) == [True, False, True, True], \
    'creator 2 has 3 creators ahead of it, so it is not fair; creators 1 and 4 tie, and ties count as fair'
assert list(is_fair([10, 3, 7, 9])) == [True, False, True, True], \
    'creator 4 overtook creator 3, but only 2 creators are ahead of creator 3, so it is still fair'

# apply your function to every run, and compare it with the `fair` column
mine = outcomes1.groupby(['run', 'timestep'])['total_followers'].transform(lambda f: is_fair(list(f)))
assert (mine == outcomes1['fair']).all(), 'is_fair disagrees with the `fair` column on some runs'
print('is_fair passes all checks')

The expected number of followers of each creator, and the share of runs in which the outcome is fair for it:

In [ ]:
plots.followers_vs_quality(outcomes1, {}, NUM_ROUNDS);
plots.fairness_vs_quality(outcomes1, {}, NUM_ROUNDS);

In [ ]:
# in how many of the 30 runs is the best creator the most followed?
final1 = outcomes1[outcomes1['timestep'] == NUM_ROUNDS]
best = final1[final1['creator'] == 0]
best.assign(most_followed=best['rank'] == 1).groupby(['rs_model', 'alpha'])['most_followed'].sum()

**Question 1.1.** Does quality determine success? Compare the recommenders: under which one is the
outcome most a matter of luck, and for which creators?

**Your answer.** *...*

## Part 2: One strategic creator

**Fake followers.** Now creator 6 (id 5) is *wealthy*: it has a budget and can buy that many fake
followers before the first round. Fake followers never follow anyone else, but the platform cannot tell
them apart from real ones: they count in the follower totals that the recommender uses and that
creators are ranked by.

**Payoffs.** After `NUM_ROUNDS` rounds, creators are ranked by followers (real + fake). The creator in
position *j* earns `TOP_PAYOFF / j` if *j* ≤ `NUM_INFLUENCERS`, and nothing otherwise. A creator who
bought followers also pays its budget:

> utility = income − budget (if it bought)

**Example.** With the defaults (`NUM_INFLUENCERS = 5`, `TOP_PAYOFF = 100`), under PA the six best creators
end with about 500, 235, 150, 112, 91 and 70 followers, so creator 6 is 6th and earns nothing. If it had
bought 25 fake followers and gained no extra real ones, it would have 95 and rank 5th: income 100 / 5 = 20,
utility 20 − 25 = −5. Whether buying pays off depends on how the recommender reacts to the extra followers.

We simulate each budget twice: once where creator 6 buys and once where it does not, with the same random seeds.

`ex.add_outcomes` computes every creator's `income`, `cost` and `utility` in every run. Creators with the
same number of followers split the prizes of the positions they occupy together.

**Prediction.** Before running the simulations: under which recommenders do you expect buying fake
followers to pay off for creator 6, and why? You will check your guess in Question 2.1.

**Your answer.** *...*

In [ ]:
BUYER = 5                   # creator 6
BUDGETS = [5, 10, 25, 50]   # fake followers

creators2, seekers2 = ex.run_experiment('part2.pkl', RECOMMENDERS,
                                        budget_profiles=[{BUYER: b} for b in BUDGETS])
# two helper columns: the buyer's budget in this game, and whether the buyer bought in this run
creators2['buyer_budget'] = creators2['budgets'].map(lambda b: dict(b)[BUYER])
creators2['buys'] = creators2['buying_creators'].map(len) > 0

outcomes2 = ex.add_outcomes(creators2, NUM_INFLUENCERS, TOP_PAYOFF)
outcomes2.head()

**Task 2.1.** Complete `gain_table(outcomes)`. It returns a table with one row per (recommender, budget)
and three columns: the buyer's expected utility when it does not buy, when it buys, and the gain from
buying (buy − don't buy). Most of it is written; fill in the two TODOs. You will reuse it in Task 2.2.

The column `buys` says whether the buyer bought in that run, and `buyer_budget` is its budget in that game.

In [ ]:
def gain_table(outcomes):
    # the buyer's rows at the end of the runs
    rows = outcomes[(outcomes['creator'] == BUYER) & (outcomes['timestep'] == NUM_ROUNDS)]

    # TODO: the buyer's mean `utility` over seeds, for every combination of
    #       rs_model, alpha, buyer_budget and buys.
    #       The pattern: df.groupby(['col1', 'col2'])['value'].mean()
    mean_utility = ...

    # one row per (recommender, budget), and one column per value of `buys` (False, True)
    table = mean_utility.unstack('buys')
    table.columns = ["don't buy", 'buy']

    # TODO: add a column 'gain': buy − don't buy
    ...

    return table.round(1)


gain_table(outcomes2)

In [ ]:
table = gain_table(outcomes2)
assert list(table.columns) == ["don't buy", 'buy', 'gain'], 'the columns should be "don\'t buy", "buy" and "gain"'
assert len(table) == len(RECOMMENDERS) * len(BUDGETS), 'there should be one row per (recommender, budget)'
assert ((table['buy'] - table["don't buy"]) - table['gain']).abs().max() <= 0.1 + 1e-9, \
    "gain should be buy − don't buy"
print('gain_table passes all checks')

The same gains as a figure. The error bars are approximate 95% intervals for the mean gain over the
paired seeds.

In [ ]:
plots.buyer_gain_vs_budget(outcomes2, BUYER, NUM_ROUNDS, TOP_PAYOFF);

**Question 2.1.**
(a) Under which recommenders is buying worth it, and at which budgets?
(b) UR ignores follower counts, so buying gives the buyer no extra visibility. Why can it still change
the buyer's income?
(c) Under ExtremePA, why does any positive budget make the buyer the most followed creator? Does that
guarantee a positive utility? (*Hint:* what does the recommender do with the follower counts in round 1?)

**Your answer.** *...*

**Question 2.2.** Under PA, how does the gain from buying change with the budget? Explain the pattern
using the buyer's position and the cost. (*Hint:* what is the largest number of **real** followers any
creator can have, and who has them at the end?)

**Your answer.** *...*

**Task 2.2.** Payoffs are computed after the simulations, so we can change them without simulating
again. Recompute the outcomes of Part 2 with *winner-take-all* payoffs, where only the most followed
creator is paid (`num_influencers = 1`), and build the gain table again.

In [ ]:
# TODO: recompute the outcomes with winner-take-all payoffs, then build the gain table
winner_takes_all = ...

In [ ]:
assert hasattr(winner_takes_all, 'loc'), 'winner_takes_all is not computed yet: complete Task 2.2 first'
assert (winner_takes_all.loc[winner_takes_all['rank'] > 1, 'income'] == 0).all(), \
    'with winner-take-all payoffs, only the most followed creator(s) earn anything'
print('winner_takes_all passes the check')

**Question 2.3.** With winner-take-all payoffs, under which recommenders does buying become more
attractive, and under which less? Explain using the buyer's position.

**Your answer.** *...*

## Part 3: Who pays for it?

We now look at how one creator's purchase changes the outcome for everyone else, comparing the runs
where creator 6 buys 25 fake followers with the runs where nobody buys: the expected followers of each
creator, the share of runs that are fair for it, how its expected income changes, and how satisfied
seekers are.

In [ ]:
plots.followers_vs_quality(outcomes2, {BUYER: 25}, NUM_ROUNDS);
plots.fairness_vs_quality(outcomes2, {BUYER: 25}, NUM_ROUNDS);
plots.income_change_by_quality(outcomes2, {BUYER: 25}, NUM_ROUNDS);
plots.seeker_dissatisfaction(seekers2, BUYER, (10, 50, 100));

Seeker dissatisfaction is the quality rank of the best creator a seeker follows, averaged over seekers
(1 means every seeker found the best creator).

**Question 3.1.** Under PA, which creators lose followers, fairness and income when creator 6 buys?
Explain the difference between creators above and below the buyer.

**Your answer.** *...*

**Question 3.2.** Are seekers better or worse off when creator 6 buys? Does your answer depend on the
recommender and on when you measure?

**Your answer.** *...*

**Question 3.3.** Under ExtremePA, when the single buyer buys, the fairness plot shows almost every creator
treated fairly. Look at the follower plot: why does the fairness measure say so, and is it a reasonable
conclusion?

**Your answer.** *...*

## Part 4: Two strategic creators

Now creators 4 and 11 (ids 3 and 10) are both wealthy, so whether buying pays for one depends on what
the other does. Each one chooses between *buy* and *don't buy*; we simulate all four combinations.
We study two games: equal budgets (25 each), and unequal budgets (creator 4 can buy 5, creator 11 can buy 50).
The expected utilities are averages over 30 seeds, so they are estimates. When two of a creator's payoffs
are nearly equal, more seeds can change which profiles are equilibria; treat such near-ties with care.

**Prediction.** Under PA with equal budgets, do you expect both creators to buy? Would they be better
or worse off than if neither could buy? You will check your guess in Question 4.1.

**Your answer.** *...*

In [ ]:
A, B = 3, 10   # creators 4 and 11
GAMES = [{A: 25, B: 25}, {A: 5, B: 50}]

creators4, seekers4 = ex.run_experiment('part4.pkl', RECOMMENDERS, budget_profiles=GAMES)
outcomes4 = ex.add_outcomes(creators4, NUM_INFLUENCERS, TOP_PAYOFF)
expected4 = ex.expected_utilities(outcomes4, NUM_ROUNDS)

# one game as a dictionary: {(A buys, B buys): (utility of A, utility of B)}
ex.payoff_table(expected4, 'general', 1, {A: 25, B: 25})

**Task 4.1.** Write `find_nash(table)`, which returns the list of pure-strategy Nash equilibria
`(a_buys, b_buys)` of a 2×2 game given in this format. A profile is an equilibrium if neither creator
can increase its utility by changing only its own choice.

In [ ]:
def find_nash(table):
    '''table: {(a_buys, b_buys): (u_a, u_b)}. Returns the list of pure Nash equilibria.'''
    # TODO
    ...

In [ ]:
# a prisoner's dilemma: both buy, although both would be better off if neither did
dilemma = {(False, False): (3, 3), (False, True): (0, 5),
           (True, False): (5, 0), (True, True): (1, 1)}
assert find_nash(dilemma) is not None, 'find_nash returns nothing yet: complete Task 4.1 first'
assert sorted(find_nash(dilemma)) == [(True, True)], \
    "prisoner's dilemma: the only equilibrium is that both buy"

# a coordination game: both want to make the same choice
coordination = {(False, False): (2, 2), (False, True): (0, 0),
                (True, False): (0, 0), (True, True): (1, 1)}
assert sorted(find_nash(coordination)) == [(False, False), (True, True)], \
    'coordination game: "neither buys" and "both buy" are both equilibria'

# A wants to match B's choice, B wants to differ from A's
pennies = {(False, False): (1, 0), (False, True): (0, 1),
           (True, False): (0, 1), (True, True): (1, 0)}
assert list(find_nash(pennies)) == [], \
    'in this game one creator always wants to switch, so there is no pure equilibrium'
print('find_nash passes all checks')

In [ ]:
for budgets in GAMES:
    print(plots.budgets_label(budgets))
    for rs_model, alpha in RECOMMENDERS:
        table = ex.payoff_table(expected4, rs_model, alpha, budgets)
        print(f'  {plots.recommender_label(rs_model, alpha):20s} equilibria: {find_nash(table)}')

In [ ]:
# the payoff tables, to support your answers (rows: creator 4, columns: creator 11)
for budgets in GAMES:
    for rs_model, alpha in RECOMMENDERS:
        print(plots.recommender_label(rs_model, alpha), '·', plots.budgets_label(budgets))
        display(plots.payoff_bimatrix(expected4, rs_model, alpha, budgets, mark_nash=False))

**Question 4.1.**
(a) For each recommender, what do the creators do in equilibrium?
(b) Are the two wealthy creators better or worse off in equilibrium than if neither could buy? Is this a
prisoner's dilemma? Compare with your prediction.
(c) With unequal budgets, does the creator with the larger budget benefit more?

**Your answer.** *...*

## Part 5: The platform's choice

The platform controls α. Use your results to answer.

**Question 5.1.** Which α would you choose to discourage creators from buying fake followers?
What does this choice cost seekers (Part 3 and Part 1)? Is there a conflict between what is good for
creators, for seekers and for the platform?

**Your answer.** *...*

**Question 5.2 (open).** Name one assumption of this model that makes buying followers particularly
effective, and how you would change the model to test it.

**Your answer.** *...*